# GeoPackage comparison: OLD vs NEW

Compares:
- **OLD:** `data/04_model_outputs/20260303/20260303_model_results.gpkg` (approved by Etienne)
- **NEW:** `data/02_processed/erosion/wocu_lgb_predictions_20260314.gpkg` (current output)

Covers:
1. Layer inventory (names, geometry types, row counts, columns)
2. Column-level diff per matching layer
3. `is_nvo` distribution analysis on `predicted_bank_positions`
4. Verdict: should `is_nvo` stay on `predicted_bank_positions`?

In [12]:
import os, sys
from pathlib import Path

_cwd = Path.cwd()
for candidate in [_cwd, *_cwd.parents]:
    if (candidate / 'src').exists():
        _backend = candidate
        break
else:
    raise RuntimeError('Could not find backend root (directory containing src/)')

if str(_backend) not in sys.path:
    sys.path.insert(0, str(_backend))
print('backend root:', _backend)

backend root: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend


In [13]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import geopandas as gpd
from pyogrio import list_layers
import src.paths as PATHS

DATA_DIR = PATHS.DATA_DIR

# OLD = original Etienne-approved output (20260303, baseline model, 2025–2035)
# Use the restored copy in the repo; falls back to Downloads if not present.
_repo_old = DATA_DIR / '04_model_outputs/20260303/20260303_model_results.gpkg'
_dl_old   = Path('/Users/admin/Downloads/20260303_model_results.gpkg')
OLD_GPKG  = _repo_old if _repo_old.exists() else _dl_old

# NEW = current ML-model output (20260314, LGB, 2026–2050)
NEW_GPKG = DATA_DIR / '04_model_outputs/20260314/wocu_lgb_predictions_20260314.gpkg'

print('OLD:', OLD_GPKG, f'({OLD_GPKG.stat().st_size / 1e6:.1f} MB)')
print('NEW:', NEW_GPKG, f'({NEW_GPKG.stat().st_size / 1e6:.1f} MB)')

OLD: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/04_model_outputs/20260303/20260303_model_results.gpkg (170.1 MB)
NEW: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/04_model_outputs/20260314/wocu_lgb_predictions_20260314.gpkg (184.7 MB)


---
## 1. Layer inventory

In [14]:
def layer_inventory(gpkg_path):
    rows = []
    for name, geom in list_layers(gpkg_path):
        gdf = gpd.read_file(gpkg_path, layer=name)
        rows.append({
            'layer':    name,
            'geom_type': str(geom) if geom else 'None',
            'n_rows':   len(gdf),
            'columns':  sorted([c for c in gdf.columns if c != 'geometry']),
        })
    return pd.DataFrame(rows).set_index('layer')

print('Loading OLD ...')
inv_old = layer_inventory(OLD_GPKG)
print('Loading NEW ...')
inv_new = layer_inventory(NEW_GPKG)

print(f'\n=== OLD: {len(inv_old)} layers ===')
for layer, row in inv_old.iterrows():
    print(f'  {layer:45s} [{row["geom_type"]:12s}]  {row["n_rows"]:>8,} rows')
    print(f'    cols: {row["columns"]}')

print(f'\n=== NEW: {len(inv_new)} layers ===')
for layer, row in inv_new.iterrows():
    print(f'  {layer:45s} [{row["geom_type"]:12s}]  {row["n_rows"]:>8,} rows')
    print(f'    cols: {row["columns"]}')

Loading OLD ...
Loading NEW ...

=== OLD: 8 layers ===
  all_lines                                     [LineString  ]   126,896 rows
    cols: ['dtm_date', 'location_id']
  erosion_vlakken_filtered                      [Polygon     ]    23,312 rows
    cols: ['area', 'dtm_version_after', 'dtm_version_before', 'erosion_volume', 'location_id', 'mean_diff_z', 'year_after', 'year_before']
  summary_scope                                 [MultiPolygon]    12,130 rows
    cols: ['dist_signalering_min', 'erosie_indicator', 'erosion_displacement', 'erosion_volume', 'missing_data', 'position_id', 'waterlichaam']
  vvr_rates_of_change                           [Polygon     ]     1,359 rows
    cols: ['consumed_most_recent', 'consumed_second_most_recent', 'distance_to_signaleringlijn_most_recent', 'distance_to_signaleringlijn_per_annum', 'distance_to_signaleringlijn_second_most_recent', 'percentage_consumed_per_annum', 'predicted_vvr_crossing_year', 'year_most_recent', 'year_second_most_recent']
 

---
## 2. Column-level diff per matching layer

In [15]:
all_layers = sorted(set(inv_old.index) | set(inv_new.index))

print('=== Column diff per layer ===')
for layer in all_layers:
    in_old = layer in inv_old.index
    in_new = layer in inv_new.index

    if not in_old:
        print(f'  {layer}: NEW ONLY')
        continue
    if not in_new:
        print(f'  {layer}: OLD ONLY')
        continue

    cols_old = set(inv_old.loc[layer, 'columns'])
    cols_new = set(inv_new.loc[layer, 'columns'])
    added    = cols_new - cols_old
    removed  = cols_old - cols_new
    rows_old = inv_old.loc[layer, 'n_rows']
    rows_new = inv_new.loc[layer, 'n_rows']
    row_diff = rows_new - rows_old

    status = '✅ identical cols' if not added and not removed else '⚠️  cols differ'
    print(f'\n  {layer}  [{status}]')
    print(f'    rows: OLD={rows_old:,}  NEW={rows_new:,}  diff={row_diff:+,}')
    if added:
        print(f'    ADDED   cols: {sorted(added)}')
    if removed:
        print(f'    REMOVED cols: {sorted(removed)}')

=== Column diff per layer ===

  all_lines  [✅ identical cols]
    rows: OLD=126,896  NEW=137,542  diff=+10,646

  distance_differences  [✅ identical cols]
    rows: OLD=10,342  NEW=11,237  diff=+895

  distances_percentiles  [✅ identical cols]
    rows: OLD=30,228  NEW=32,866  diff=+2,638

  erosion_vlakken_filtered  [✅ identical cols]
    rows: OLD=23,312  NEW=25,194  diff=+1,882

  predicted_bank_positions  [⚠️  cols differ]
    rows: OLD=115,324  NEW=202,350  diff=+87,026
    ADDED   cols: ['fid']
  signaleringslijn: NEW ONLY

  summary_scope  [⚠️  cols differ]
    rows: OLD=12,130  NEW=16,772  diff=+4,642
    ADDED   cols: ['estimate_reliability_height_model']

  vvr_all_results  [✅ identical cols]
    rows: OLD=4,018  NEW=4,174  diff=+156

  vvr_rates_of_change  [✅ identical cols]
    rows: OLD=1,359  NEW=1,410  diff=+51


---
## 3. `is_nvo` distribution on `predicted_bank_positions`

Key questions:
- Is every location_id consistently either NVO or non-NVO across all prediction years?
- If yes, `is_nvo` adds no per-row information — it belongs only on the scope/VVR layer.

In [16]:
print('Reading predicted_bank_positions from OLD ...')
pbp_old = gpd.read_file(OLD_GPKG, layer='predicted_bank_positions')
print(f'  {len(pbp_old):,} rows, {pbp_old["location_id"].nunique():,} unique locations')
print(f'  is_nvo present: {"is_nvo" in pbp_old.columns}')

if 'is_nvo' in pbp_old.columns:
    print(f'\n  is_nvo value counts:')
    print(pbp_old['is_nvo'].value_counts().to_string())

    # Check: does any location_id have MIXED is_nvo values across years?
    nvo_per_loc = pbp_old.groupby('location_id')['is_nvo'].nunique()
    mixed = nvo_per_loc[nvo_per_loc > 1]
    print(f'\n  Locations with MIXED is_nvo across years: {len(mixed)}')
    if len(mixed):
        print('  (these locations have inconsistent is_nvo — keeping it on pbp matters)')
        print(mixed)
    else:
        print('  → All locations have consistent is_nvo across all prediction years.')
        print('  → is_nvo is REDUNDANT on predicted_bank_positions.')
        n_per_loc = pbp_old.groupby('location_id').size()
        print(f'  → Each location has {n_per_loc.unique()} prediction rows (should all be the same).')

Reading predicted_bank_positions from OLD ...
  115,324 rows, 10,484 unique locations
  is_nvo present: True

  is_nvo value counts:
is_nvo
False    88627
True     26697

  Locations with MIXED is_nvo across years: 0
  → All locations have consistent is_nvo across all prediction years.
  → is_nvo is REDUNDANT on predicted_bank_positions.
  → Each location has [11] prediction rows (should all be the same).


In [17]:
# Same check on NEW (is_nvo might be missing due to pyogrio bug — check raw sqlite)
import sqlite3

con = sqlite3.connect(NEW_GPKG)
cols_new_pbp = [r[1] for r in con.execute("PRAGMA table_info('predicted_bank_positions')")]
con.close()

print('Columns actually stored in NEW predicted_bank_positions (SQLite):')
print(cols_new_pbp)

Columns actually stored in NEW predicted_bank_positions (SQLite):
['fid', 'geom', 'location_id', 'year', 'predicted_dist_m', 'velocity_m_per_yr', 'is_nvo']


---
## 4. Where is `is_nvo` used in the pipeline?

Summary of findings from code search:

| Location | Usage | Still needed on predicted_bank_positions? |
|---|---|---|
| `20260303_create_region_split.ipynb` (archived) | Reads `is_nvo` from `predicted_bank_positions` to populate `region_split.parquet` | **No — this notebook is archived** |
| `20260305_preprocess_region_split.ipynb` | Loads `is_nvo` from existing `region_split.parquet` directly | **No** |
| `20260311_preprocess_region_split_v2.ipynb` | Carries `is_nvo` over from `split_v1.parquet` | **No** |
| `20260311/20260312_model_comparison.ipynb` | `is_nvo` as a model feature from `features_df` (parquet) | **No** |
| `archive/20260312_validate_rebuilt_output.ipynb` | Reads `is_nvo` from `predicted_bank_positions` to filter NVO locations | **Archived — no** |
| `src/erosion/plot_utils.py` | Does NOT use `is_nvo` | **No** |
| `src/erosion/export.py` | Writes `is_nvo` to the layer — no downstream reader | Depends on QGIS use |

**Verdict:** `is_nvo` is NOT used anywhere in the current active pipeline that reads it from `predicted_bank_positions`. 
The only reason to keep it is for **QGIS styling** (colour NVO predictions differently from non-NVO).
Since every `location_id` has the same `is_nvo` value for all years, the information is redundant with the `vvr_rates_of_change` layer (NVO regions are those covered by a VVR polygon).

**Recommendation:** Keep `is_nvo` on `predicted_bank_positions` for QGIS convenience (avoids a spatial join in QGIS), but fix the pyogrio bool-as-geometry bug by casting to `int` before writing.

In [18]:
# How many unique locations are in OLD but not in NEW?
pbp_new = gpd.read_file(NEW_GPKG, layer='predicted_bank_positions')

locs_old = set(pbp_old['location_id'].unique())
locs_new = set(pbp_new['location_id'].unique())

print(f'OLD unique locations: {len(locs_old):,}')
print(f'NEW unique locations: {len(locs_new):,}')
print(f'In OLD but not NEW:  {len(locs_old - locs_new):,}')
print(f'In NEW but not OLD:  {len(locs_new - locs_old):,}')

# Break down missing locations by quality using the 20260314 region_split
_rs_paths = [
    DATA_DIR / '03_features/20260314/region_split.parquet',
    DATA_DIR / '02_processed/erosion/region_split.parquet',
]
region_split = None
for p in _rs_paths:
    if p.exists():
        region_split = pd.read_parquet(p)
        print(f'\nUsing region_split from: {p.name}')
        break

if region_split is not None and 'quality' in region_split.columns:
    missing_in_new = region_split[region_split.index.isin(locs_old - locs_new)]
    print(f'Quality breakdown for locations in OLD but not NEW ({len(missing_in_new):,} found in split):')
    print(missing_in_new['quality'].value_counts().to_string())
    not_in_split = len(locs_old - locs_new) - len(missing_in_new)
    if not_in_split:
        print(f'  ({not_in_split:,} not found in region_split — likely inference_only or MISSING_DATA)')
elif region_split is not None:
    print('region_split columns:', list(region_split.columns))

OLD unique locations: 10,484
NEW unique locations: 8,094
In OLD but not NEW:  2,390
In NEW but not OLD:  0

Using region_split from: region_split.parquet
Quality breakdown for locations in OLD but not NEW (0 found in split):
Series([], )
  (2,390 not found in region_split — likely inference_only or MISSING_DATA)
